In [3]:
# ============================================================
# USE YOUR EXISTING TEXT MODEL TO ANALYZE IMAGES
# Image → OCR → Your Text Model → Prediction
# ============================================================

!pip install pytesseract pillow opencv-python --quiet
!apt-get update -qq && apt-get install -y tesseract-ocr -qq

import pytesseract
from PIL import Image
import cv2
import numpy as np
import pickle

# ============================================================
# STEP 1: LOAD YOUR EXISTING MODEL AND VECTORIZER
# ============================================================

# Load your trained model (the one you already have)
model = pickle.load(open('../models/cyberbullying_model.pkl', 'rb'))
vectorizer = pickle.load(open('../models/tfidf_vectorizer.pkl', 'rb'))

print("✅ Your text model loaded!")


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


✅ Your text model loaded!


In [4]:
# ============================================================
# STEP 2: OCR FUNCTION - Extract Text from Image
# ============================================================

def extract_text_from_image(image_path):
    """
    Extract text from screenshot using OCR
    """
    # Load image
    image = Image.open(image_path)
    
    # Convert to array for preprocessing
    img_array = np.array(image)
    
    # Convert to grayscale
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_array
    
    # Improve OCR accuracy with preprocessing
    denoised = cv2.fastNlMeansDenoising(gray)
    _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # OCR
    text = pytesseract.image_to_string(thresh)
    
    return text.strip()

In [5]:
# ============================================================
# STEP 3: ANALYZE IMAGE WITH YOUR EXISTING MODEL
# ============================================================

def analyze_image_with_text_model(image_path):
    """
    Complete pipeline: Image → OCR → Your Text Model → Result
    """
    print(f"\n📸 Processing image: {image_path}")
    
    # Step 1: Extract text from image (OCR)
    print("🔍 Extracting text with OCR...")
    extracted_text = extract_text_from_image(image_path)
    
    if not extracted_text:
        print("❌ No text found in image")
        return None
    
    print(f"✅ Extracted {len(extracted_text)} characters")
    print("-" * 60)
    print("EXTRACTED TEXT:")
    print(extracted_text[:500] + "..." if len(extracted_text) > 500 else extracted_text)
    print("-" * 60)
    
    # Step 2: Split into individual messages
    lines = [line.strip() for line in extracted_text.split('\n') if line.strip()]
    
    # Step 3: Analyze each line with YOUR existing model
    print("\n🔍 Analyzing with your text model...")
    
    bullying_messages = []
    safe_messages = []
    total_bullying_score = 0
    
    for line in lines:
        if len(line) < 3:  # Skip very short lines
            continue
        
        # Use YOUR model to predict
        line_tfidf = vectorizer.transform([line.lower()])
        prediction = model.predict(line_tfidf)[0]
        probabilities = model.predict_proba(line_tfidf)[0]
        
        bullying_prob = probabilities[1] * 100
        total_bullying_score += bullying_prob
        
        result = {
            'text': line,
            'prediction': "BULLYING" if prediction == 1 else "SAFE",
            'bullying_probability': bullying_prob,
            'safe_probability': probabilities[0] * 100
        }
        
        if prediction == 1:
            bullying_messages.append(result)
        else:
            safe_messages.append(result)
    
    # Step 4: Calculate overall results
    total_messages = len(bullying_messages) + len(safe_messages)
    
    if total_messages == 0:
        print("❌ No valid messages to analyze")
        return None
    
    avg_bullying_score = total_bullying_score / total_messages
    bullying_ratio = (len(bullying_messages) / total_messages) * 100
    
    # Determine overall status
    if bullying_ratio > 30 or avg_bullying_score > 50:
        status = "🚨 CYBERBULLYING DETECTED IN IMAGE"
        risk = "HIGH"
    elif bullying_ratio > 10 or avg_bullying_score > 30:
        status = "⚠️ SUSPICIOUS CONTENT"
        risk = "MODERATE"
    else:
        status = "✅ IMAGE APPEARS SAFE"
        risk = "LOW"
    
    # Display results
    print("\n" + "=" * 60)
    print("📊 ANALYSIS RESULTS")
    print("=" * 60)
    print(f"Total messages found: {total_messages}")
    print(f"🔴 Bullying messages: {len(bullying_messages)}")
    print(f"🟢 Safe messages: {len(safe_messages)}")
    print(f"📈 Bullying ratio: {bullying_ratio:.1f}%")
    print(f"📉 Avg bullying score: {avg_bullying_score:.1f}%")
    print(f"\n🎯 OVERALL: {status}")
    print(f"⚖️ Risk Level: {risk}")
    
    # Show flagged messages
    if bullying_messages:
        print(f"\n🚩 FLAGGED MESSAGES:")
        for msg in sorted(bullying_messages, key=lambda x: x['bullying_probability'], reverse=True)[:5]:
            print(f"\n   {msg['bullying_probability']:.1f}% bullying:")
            print(f"   \"{msg['text'][:80]}...\"")
    
    print("=" * 60)
    
    return {
        'status': status,
        'risk_level': risk,
        'bullying_ratio': bullying_ratio,
        'bullying_messages': bullying_messages,
        'safe_messages': safe_messages
    }


In [13]:
# ============================================================
# STEP 4: LOAD AND ANALYZE LOCAL IMAGE
# ============================================================

import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


print("=" * 60)
print("📂 ANALYZING LOCAL WHATSAPP SCREENSHOT")
print("=" * 60)

# Replace this with your actual file path
image_path = "../Screenshot_20260322-033134.jpg"

# Run your existing function
result = analyze_image_with_text_model(image_path)

print(result)
# ============================================================
# STEP 5: TEST WITH HARDCODED PATH (Alternative)
# ============================================================

# If you have image saved locally:
# analyze_image_with_text_model("my_whatsapp_screenshot.png")

📂 ANALYZING LOCAL WHATSAPP SCREENSHOT

📸 Processing image: ../Screenshot_20260322-033134.jpg
🔍 Extracting text with OCR...
✅ Extracted 341 characters
------------------------------------------------------------
EXTRACTED TEXT:
3:31AM © @ Oe 2 Sls s

e +234905 443...
€ - 3:30 AM Cr %

10:26 AM W/

Today

You what's UP 3-35 any

You're a fucking fool 3.45 ayy

i 22?
How stupid can you be??? 4.45 any

MIE Kill you 5.35 an
What do you mean?? 3.31 amy
Don't let me kill you. 3.3, ayy

| told you what happened and you're

angry with me 3:31 AM ¥

@ Message vo ©
2)

= O
------------------------------------------------------------

🔍 Analyzing with your text model...

📊 ANALYSIS RESULTS
Total messages found: 16
🔴 Bullying messages: 5
🟢 Safe messages: 11
📈 Bullying ratio: 31.2%
📉 Avg bullying score: 48.7%

🎯 OVERALL: 🚨 CYBERBULLYING DETECTED IN IMAGE
⚖️ Risk Level: HIGH

🚩 FLAGGED MESSAGES:

   99.9% bullying:
   "How stupid can you be??? 4.45 any..."

   99.8% bullying:
   "You're a fucking foo